In [ ]:
!env CUBLAS_WORKSPACE_CONFIG=":4096:8" PYTHONHASHSEED="42" python get_embedding.py

In [ ]:
!env CUBLAS_WORKSPACE_CONFIG=":4096:8" PYTHONHASHSEED="42" python run_baseline.py

In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
import contextlib
import io
from tqdm import tqdm
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import shap
import logging

logging.getLogger("shap").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# =========================
# 1. Configuration & Paths
# =========================
OIL_CSV = "data/rv.csv"
EMBEDDING_CSV = "gnn_embeddings/gnn_oil_context_full_mean_normT_fusT_from_20150401_cameo_ge10.csv"
SYMBOL = "CL_c1"
BASELINE_DIR = "baseline_results"
OUTPUT_DIR = "results"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

EVAL_START_DATE = "2018-01-01"
EVAL_END_DATE = "2026-04-28"
ROLLING_WINDOW_SIZE = 1000

TASKS = [
    ("Target_1D",  1),
    ("Target_1W",  5),
    ("Target_2W", 10),
    ("Target_3W", 15),
    ("Target_1M", 22),
]

PCA_DIMS_TO_TEST = [1]
BASELINES_TO_COMPARE = ["HAR", "HAR-J", "HAR-OVX", "HAR-OVX-GPR"]

# =========================
# 2. Metric Functions
# =========================
def qlike_loss_np(y_true, y_pred):
    eps = 1e-6
    y_pred = np.clip(y_pred, eps, None)
    y_true = np.clip(y_true, eps, None)
    ratio = y_true / y_pred
    return np.mean(ratio - np.log(ratio) - 1)

def calculate_r2_oos(y_true, y_pred, y_bench):
    mse_model = np.mean((y_true - y_pred) ** 2)
    mse_bench = np.mean((y_true - y_bench) ** 2)
    return 1 - (mse_model / (mse_bench + 1e-10))

def set_seed(seed=42):
    np.random.seed(seed)

def subtract_trading_days(date: pd.Timestamp, n_trading_days: int) -> pd.Timestamp:
    result = np.busday_offset(date.date(), -n_trading_days, roll='preceding')
    return pd.Timestamp(result)

# =========================
# 4. Data Preparation
# =========================
def prepare_data_raw(oil_csv, emb_csv, symbol, horizons_map):
    print("Preparing Raw Data (HAR + OVX + GPR + GNN)...")

    data = pd.read_csv(oil_csv)
    df = data[data["symbol"] == symbol].sort_values(by="date", ascending=True).copy()
    df["date"] = pd.to_datetime(df["date"])
    df.set_index("date", inplace=True)

    ovx_data = data[data["symbol"] == "OVX"].copy()
    if not ovx_data.empty:
        ovx_data["date"] = pd.to_datetime(ovx_data["date"])
        ovx_data = ovx_data.set_index("date")[["rv"]].rename(columns={"rv": "OVX"})
        df = df.join(ovx_data, how="left")
        df["OVX"] = df["OVX"].ffill()
        df["log_OVX"] = np.log(df["OVX"] + 1e-10)
    else:
        df["log_OVX"] = 0.0

    gpr_df = data[data["symbol"] == "GPR"].copy()
    if not gpr_df.empty:
        gpr_df["date"] = pd.to_datetime(gpr_df["date"])
        gpr_df = gpr_df.set_index("date")[["rv"]].rename(columns={"rv": "GPR"})
        df = df.join(gpr_df, how="left")
        df["GPR"] = df["GPR"].ffill()
        df["log_GPR"] = np.log(df["GPR"] + 1e-10)
    else:
        df["log_GPR"] = 0.0

    df["RV_d"] = df["rv"]
    df["RV_w"] = df["rv"].rolling(window=5).mean()
    df["RV_m"] = df["rv"].rolling(window=22).mean()
    df["log_RV_d"] = np.log(df["RV_d"] + 1e-10)
    df["log_RV_w"] = np.log(df["RV_w"] + 1e-10)
    df["log_RV_m"] = np.log(df["RV_m"] + 1e-10)

    gnn_cols = []
    if os.path.exists(emb_csv):
        emb_df = pd.read_csv(emb_csv)
        emb_df["date"] = pd.to_datetime(emb_df["date"])
        emb_df.set_index("date", inplace=True)
        gnn_cols = [c for c in emb_df.columns if "gnn_" in c]
        df = df.join(emb_df[gnn_cols], how="left")
        df[gnn_cols] = df[gnn_cols].ffill().fillna(0).shift(1)
    else:
        raise FileNotFoundError(f"Embedding file not found: {emb_csv}")

    next_day_rv = df["rv"].shift(-1)
    for name, days in horizons_map.items():
        indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=days)
        df[name] = next_day_rv.rolling(window=indexer).mean()

    features_to_check = ["log_RV_d", "log_RV_w", "log_RV_m", "log_OVX"] + gnn_cols
    df.dropna(subset=features_to_check, inplace=True)

    return df, gnn_cols

# =========================
# 5. Rolling PCA Forecast Logic
# =========================
def run_rolling_pca_forecast(target_col, horizon_days, n_components, start_date, end_date):

    horizons_map = {target_col: horizon_days}
    df, gnn_cols = prepare_data_raw(OIL_CSV, EMBEDDING_CSV, SYMBOL, horizons_map)

    har_features = ["log_RV_d", "log_RV_w", "log_RV_m", "log_OVX"]
    feature_names = har_features + [f"PC{k+1}" for k in range(n_components)]

    min_train_size = 100
    train_end_idx = min_train_size + horizon_days

    y_true, y_pred, y_bench = [], [], []
    dates = []
    shap_records = []

    scaler_har = StandardScaler()
    scaler_gnn = StandardScaler()
    model = Ridge(alpha=1.0, random_state=42)

    for i in tqdm(range(train_end_idx, len(df)),
                  desc=f"PCA({n_components}) {target_col}", leave=False):
        curr_date = df.index[i]

        valid_train_end = i - horizon_days
        train_start_idx = max(0, valid_train_end - ROLLING_WINDOW_SIZE)

        train_slice = df.iloc[train_start_idx:valid_train_end].copy()
        train_slice.dropna(subset=[target_col], inplace=True)

        if len(train_slice) < min_train_size:
            continue

        test_slice = df.iloc[i:i+1]

        X_har_train_raw = train_slice[har_features].values
        X_har_test_raw  = test_slice[har_features].values
        X_gnn_train_raw = train_slice[gnn_cols].values
        X_gnn_test_raw  = test_slice[gnn_cols].values
        y_train_log     = np.log(train_slice[target_col].values + 1e-10)

        X_gnn_train_scaled = scaler_gnn.fit_transform(X_gnn_train_raw)
        pca = PCA(n_components=n_components, random_state=42)
        X_gnn_train_pca   = pca.fit_transform(X_gnn_train_scaled)
        X_gnn_test_scaled = scaler_gnn.transform(X_gnn_test_raw)
        X_gnn_test_pca    = pca.transform(X_gnn_test_scaled)

        X_har_train_scaled = scaler_har.fit_transform(X_har_train_raw)
        X_har_test_scaled  = scaler_har.transform(X_har_test_raw)

        X_train = np.hstack([X_har_train_scaled, X_gnn_train_pca])
        X_test  = np.hstack([X_har_test_scaled,  X_gnn_test_pca])

        model.fit(X_train, y_train_log)

        # ── SHAP：interventional 模式，無 verbose 輸出 ──
        with contextlib.redirect_stdout(io.StringIO()), \
             contextlib.redirect_stderr(io.StringIO()):
            explainer = shap.LinearExplainer(
                model, X_train,
                feature_perturbation="interventional"
            )
            sv = explainer.shap_values(X_test)  # shape: [1, n_features]

        record = {"date": curr_date}
        for fname, sval in zip(feature_names, sv[0]):
            record[f"shap_{fname}"] = sval
        shap_records.append(record)

        pred_val  = np.exp(model.predict(X_test)[0])
        bench_val = train_slice[target_col].mean()

        y_true.append(test_slice[target_col].values[0])
        y_pred.append(pred_val)
        y_bench.append(bench_val)
        dates.append(curr_date)

    res_df_full = pd.DataFrame({
        "date": dates,
        "Actual": y_true,
        "Pred": y_pred,
        "Bench": y_bench
    })

    shap_df = pd.DataFrame(shap_records)

    if len(res_df_full) == 0:
        return {
            "Horizon": target_col, "PCA_Dim": n_components,
            "QLIKE": np.nan, "R2": np.nan, "R2_OOS": np.nan, "MSE": np.nan
        }, pd.DataFrame(), shap_df

    ts_start = pd.Timestamp(start_date)
    ts_end   = pd.Timestamp(end_date)
    mask     = (res_df_full['date'] >= ts_start) & (res_df_full['date'] <= ts_end)
    eval_df  = res_df_full.loc[mask]
    valid_eval_df = eval_df.dropna(subset=["Actual"])

    if len(valid_eval_df) < 2:
        return {
            "Horizon": target_col, "PCA_Dim": n_components,
            "QLIKE": np.nan, "R2": np.nan, "R2_OOS": np.nan, "MSE": np.nan
        }, res_df_full[["date", "Actual", "Pred"]].copy(), shap_df

    y_true_eval  = valid_eval_df["Actual"].values
    y_pred_eval  = valid_eval_df["Pred"].values
    y_bench_eval = valid_eval_df["Bench"].values

    result_dict = {
        "Horizon": target_col,
        "PCA_Dim": n_components,
        "QLIKE":   qlike_loss_np(y_true_eval, y_pred_eval),
        "R2":      r2_score(y_true_eval, y_pred_eval),
        "R2_OOS":  calculate_r2_oos(y_true_eval, y_pred_eval, y_bench_eval),
        "MSE":     mean_squared_error(y_true_eval, y_pred_eval)
    }

    res_df_out = res_df_full[["date", "Actual", "Pred"]].copy()

    return result_dict, res_df_out, shap_df

# =========================
# 6. Evaluation Function (For Baselines)
# =========================
def evaluate_csv_baseline(file_path, start_date, end_date):
    try:
        df = pd.read_csv(file_path)
        df['date'] = pd.to_datetime(df['date'])
        mask = (df['date'] >= start_date) & (df['date'] <= end_date)
        df_eval = df.loc[mask].copy()
        df_eval.dropna(subset=['Actual', 'Pred'], inplace=True)
        if len(df_eval) < 2:
            return None
        y_true = df_eval['Actual'].values
        y_pred = df_eval['Pred'].values
        return {
            "QLIKE": qlike_loss_np(y_true, y_pred),
            "R2":    r2_score(y_true, y_pred),
            "MSE":   mean_squared_error(y_true, y_pred)
        }
    except Exception:
        return None

# =========================
# 7. Main Pipeline
# =========================
set_seed(42)

raw_data = pd.read_csv(OIL_CSV)
actual_last_date_oil = pd.to_datetime(raw_data['date']).max()

emb_data = pd.read_csv(EMBEDDING_CSV)
actual_last_date_gnn = pd.to_datetime(emb_data['date']).max()

actual_last_date = min(actual_last_date_oil, actual_last_date_gnn)
max_horizon_days = max(d for _, d in TASKS)

GLOBAL_OOS_END = min(
    pd.Timestamp(EVAL_END_DATE),
    subtract_trading_days(actual_last_date, max_horizon_days)
)

print(f"========================================================")
print(f"   Strict Rolling PCA Analysis (Dims: {PCA_DIMS_TO_TEST})")
print(f"   Window Type: Fixed Rolling ({ROLLING_WINDOW_SIZE})")
print(f"   OIL data last date   : {actual_last_date_oil.date()}")
print(f"   GNN data last date   : {actual_last_date_gnn.date()}")
print(f"   Effective last date  : {actual_last_date.date()}")
print(f"   Max horizon (trading): {max_horizon_days} days")
print(f"   Evaluation Start     : {EVAL_START_DATE}")
print(f"   Evaluation End       : {GLOBAL_OOS_END.date()}")
print(f"========================================================\n")

all_results = []

for target_name, days in TASKS:
    print(f">>> Analyzing {target_name}...")

    for k in PCA_DIMS_TO_TEST:
        res, res_df, shap_df = run_rolling_pca_forecast(
            target_name, days, k, EVAL_START_DATE, GLOBAL_OOS_END
        )
        res["Model"] = f"HAR-OVX+RollPCA({k})"
        all_results.append(res)

        if not res_df.empty:
            fname = os.path.join(
                OUTPUT_DIR,
                f"full_history_pred_HAR-OVX+RollPCA({k})_{target_name}.csv"
            )
            res_df.to_csv(fname, index=False)

        if not shap_df.empty:
            shap_fname = os.path.join(
                OUTPUT_DIR,
                f"shap_timeseries_HAR-OVX+RollPCA({k})_{target_name}.csv"
            )
            shap_df.to_csv(shap_fname, index=False)
            print(f"    SHAP saved → {shap_fname}  ({len(shap_df)} rows)")

    for model_name in BASELINES_TO_COMPARE:
        baseline_file = os.path.join(
            BASELINE_DIR,
            f"full_history_pred_{model_name}_{target_name}.csv"
        )
        if os.path.exists(baseline_file):
            base_metrics = evaluate_csv_baseline(
                baseline_file, EVAL_START_DATE, GLOBAL_OOS_END
            )
            if base_metrics:
                base_res = {
                    "Horizon": target_name,
                    "PCA_Dim": 0,
                    "Model":   f"{model_name} (Base)",
                    "R2_OOS":  np.nan,
                    **base_metrics
                }
                all_results.append(base_res)

if all_results:
    final_df = pd.DataFrame(all_results)

    print("\n========================================================")
    print("   Rolling PCA vs Baselines Report")
    print("========================================================")

    pivot_df = final_df.pivot(index="Horizon", columns="Model", values="QLIKE")
    base_cols = [f"{m} (Base)" for m in BASELINES_TO_COMPARE]
    pca_cols  = [f"HAR-OVX+RollPCA({k})" for k in PCA_DIMS_TO_TEST]
    sorted_cols = [c for c in base_cols + pca_cols if c in pivot_df.columns]
    pivot_df = pivot_df[sorted_cols]

    print("\n--- QLIKE Matrix (Lower is Better) ---")
    pd.options.display.float_format = "{:,.6f}".format
    print(pivot_df)

    print("\n--- Best Model per Horizon ---")
    for target_name, _ in TASKS:
        subset = final_df[final_df["Horizon"] == target_name]
        if not subset.empty:
            best_row = subset.loc[subset["QLIKE"].idxmin()]
            print(f"{target_name}: Best={best_row['Model']} "
                  f"(QLIKE={best_row['QLIKE']:.6f})")

    out_path = os.path.join(OUTPUT_DIR, "rolling_pca_sensitivity_results.csv")
    final_df.to_csv(out_path, index=False)
    print(f"\nFull results saved to '{out_path}'")
else:
    print("No results generated.")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import matplotlib.transforms as transforms

warnings.filterwarnings("ignore")


RESULTS_DIR = "results"
BASELINE_DIR = "baseline_results"
FIGS_DIR = "figs"

EVAL_START_DATE = "2018-01-01"
EVAL_END_DATE = "2026-04-28"

TARGETS = [
    "Target_1D", "Target_1W", "Target_2W", "Target_3W",
    "Target_1M"
]

FILE_PREFIX = "full_history_pred_"

CHALLENGER_MODEL = "HAR-OVX+RollPCA(1)"
CHALLENGER_LABEL = "Proposed"
BASELINE_MODEL = "HAR-OVX-GPR"

SMOOTH_WINDOW = 22

MAJOR_EVENTS = {
    "COVID-19": "2020-03-11",
    "Russia-Ukraine War": "2022-02-24",
    "US-Iran War": "2026-02-27"
}


TARGET_TO_FILENAME = {
    "Target_1D": "1d.png",
    "Target_1W": "1w.png",
    "Target_2W": "2w.png",
    "Target_3W": "3w.png",
    "Target_1M": "1m.png"
}


def qlike_loss_series(y_true, y_pred):
    eps = 1e-6
    y_pred = np.clip(y_pred, eps, None)
    y_true = np.clip(y_true, eps, None)
    return y_true / y_pred - np.log(y_true / y_pred) - 1


def plot_regime_performance_individual():
    plt.style.use('seaborn-v0_8-whitegrid')
    sns.set_context("paper", font_scale=1.2)

    os.makedirs(FIGS_DIR, exist_ok=True)

    print("Generating Individual Regime Performance Plots...")
    print(f"(Challenger: {CHALLENGER_MODEL} -> {CHALLENGER_LABEL} vs Baseline: {BASELINE_MODEL})\n")

    year_fmt = mdates.DateFormatter('%Y')
    year_loc = mdates.YearLocator()

    for target in TARGETS:
        print(f"  Processing {target}...")

        loss_dict = {}
        actual_vol = None

        challenger_path = os.path.join(
            RESULTS_DIR, f"{FILE_PREFIX}{CHALLENGER_MODEL}_{target}.csv"
        )
        baseline_path = os.path.join(
            BASELINE_DIR, f"{FILE_PREFIX}{BASELINE_MODEL}_{target}.csv"
        )

        file_map = {
            CHALLENGER_MODEL: challenger_path,
            BASELINE_MODEL: baseline_path
        }

        for m_name, f_path in file_map.items():
            if not os.path.exists(f_path):
                print(f"    [Warning] File not found: {f_path}")
                continue

            df = pd.read_csv(f_path)
            df['date'] = pd.to_datetime(df['date'])
            df = df.drop_duplicates(subset=['date']).set_index('date').sort_index()

            mask = (df.index >= EVAL_START_DATE) & (df.index <= EVAL_END_DATE)
            df = df.loc[mask]

            if df.empty:
                continue

            df_clean = df.dropna(subset=['Actual', 'Pred'])

            loss_dict[m_name] = pd.Series(
                qlike_loss_series(df_clean['Actual'].values, df_clean['Pred'].values),
                index=df_clean.index
            )

            if actual_vol is None or m_name == CHALLENGER_MODEL:
                actual_vol = df_clean['Actual']

        if len(loss_dict) < 2:
            print(f"    [Skip] Data Missing for {target}")
            continue

        common_index = loss_dict[CHALLENGER_MODEL].index.intersection(loss_dict[BASELINE_MODEL].index)
        if len(common_index) == 0:
            print(f"    [Skip] No overlapping dates for {target}")
            continue

        challenger_loss = loss_dict[CHALLENGER_MODEL].loc[common_index]
        baseline_loss = loss_dict[BASELINE_MODEL].loc[common_index]
        actual_aligned = actual_vol.loc[common_index]

        diff = baseline_loss - challenger_loss
        diff_smooth = diff.rolling(window=SMOOTH_WINDOW, center=True).mean()

        plot_data = pd.DataFrame({
            'Actual': actual_aligned,
            'Diff': diff_smooth
        }).dropna()

        if plot_data.empty:
            print(f"    [Skip] No overlapping data after smoothing for {target}")
            continue

        dates = plot_data.index
        y_vals = plot_data['Actual'].values
        diff_vals = plot_data['Diff'].values

        fig, ax = plt.subplots(figsize=(15, 4))

        ax.plot(dates, y_vals, color='black', lw=1.2, label='Realized Volatility', alpha=0.8)

        ax.fill_between(
            dates, 0, y_vals, where=(diff_vals > 0),
            color='green', alpha=0.3, label=f'{CHALLENGER_LABEL} Wins', interpolate=True
        )

        ax.fill_between(
            dates, 0, y_vals, where=(diff_vals <= 0),
            color='red', alpha=0.3, label=f'{BASELINE_MODEL} Wins', interpolate=True
        )

        for event_name, event_date in MAJOR_EVENTS.items():
            event_dt = pd.to_datetime(event_date)
            if dates.min() <= event_dt <= dates.max():
                ax.axvline(x=event_dt, color='blue', linestyle='--', linewidth=1.2, alpha=0.6)
                trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
                ax.text(
                    event_dt, 0.95, f' {event_name}', transform=trans,
                    color='blue', fontsize=10, fontweight='bold', alpha=0.8,
                    verticalalignment='top', horizontalalignment='left'
                )

        horizon_label = target.replace("Target_", "")
        ax.set_title(
            f"{horizon_label} Performance Regime (vs {BASELINE_MODEL})",
            fontsize=14, fontweight='bold', loc='left'
        )
        ax.set_ylabel("Realized Volatility")
        ax.grid(True, linestyle='--', alpha=0.5)

        ax.set_xlim(dates.min(), dates.max())
        ax.tick_params(axis='x', which='both', labelbottom=True)
        ax.xaxis.set_major_formatter(year_fmt)
        ax.xaxis.set_major_locator(year_loc)

        ax.legend(loc='upper left', bbox_to_anchor=(0.01, 0.85), frameon=True, framealpha=0.9)

        plt.tight_layout()

        save_name = TARGET_TO_FILENAME[target]
        save_path = os.path.join(FIGS_DIR, save_name)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"    [Saved] {save_path}")

        plt.show()
        plt.close(fig)

if __name__ == "__main__":
    plot_regime_performance_individual()

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.sandwich_covariance import cov_hac
import warnings

warnings.filterwarnings("ignore")

RESULTS_DIR = "results"
BASELINE_DIR = "baseline_results"

EVAL_START_DATE = "2018-01-01"
EVAL_END_DATE = "2026-04-28"  


TARGETS = [
    "Target_1D", "Target_1W", "Target_2W", "Target_3W", 
    "Target_1M"
]

FILE_PREFIX = "full_history_pred_"

SELECTED_MODELS = [
    "HAR",
    "HAR-J",
    "HAR-OVX",
    "HAR-OVX-GPR",
    "HAR-OVX+RollPCA(1)"
]


def qlike_loss_series(y_true, y_pred):
    """計算 QLIKE Loss"""
    eps = 1e-6
    y_pred = np.clip(y_pred, eps, None)
    y_true = np.clip(y_true, eps, None)
    ratio = y_true / y_pred
    loss = ratio - np.log(ratio) - 1
    return loss


def dm_test(loss_a, loss_b, lags=5):
    """
    Diebold-Mariano Test (使用 Newey-West HAC 標準誤)
    """
    d = loss_a - loss_b
    mean_diff = d.mean()
    
    if abs(mean_diff) < 1e-10:
        return 0.0, 1.0
    
    try:
        d_array = d.values.reshape(-1, 1)
        var_d = cov_hac(d_array, nlags=lags, maxlags=None)[0, 0]
        se_d = np.sqrt(var_d / len(d))
    except Exception:
        se_d = d.std() / np.sqrt(len(d))
    
    dm_stat = mean_diff / se_d
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    
    return dm_stat, p_value



if __name__ == "__main__":
    

    max_available_dates = []
    
    for target in TARGETS:
        for model_name in SELECTED_MODELS:
            if model_name == "HAR-OVX+RollPCA(1)":
                f_path = os.path.join(RESULTS_DIR, f"{FILE_PREFIX}{model_name}_{target}.csv")
            else:
                f_path = os.path.join(BASELINE_DIR, f"{FILE_PREFIX}{model_name}_{target}.csv")
                
            if os.path.exists(f_path):
                df_check = pd.read_csv(f_path, usecols=['date'])
                max_available_dates.append(pd.to_datetime(df_check['date']).max())
                
    if max_available_dates:
  
        effective_end_date = min(max_available_dates)
        effective_end_date = min(effective_end_date, pd.Timestamp(EVAL_END_DATE))
    else:
        effective_end_date = pd.Timestamp(EVAL_END_DATE)

    print(f"========================================================")
    print(f"   Pairwise Diebold-Mariano Tests (Uniform Cut-off)")
    print(f"   Evaluation Start : {EVAL_START_DATE}")
    print(f"   Evaluation End   : {effective_end_date.date()} (Strictly Aligned)")
    print(f"   Models Included  : {SELECTED_MODELS}")
    print(f"========================================================\n")

    all_dm_results = []

    for target in TARGETS:
        print(f">>> Processing {target} ...")
        loss_dict = {}
        
    
        for model_name in SELECTED_MODELS:
            if model_name == "HAR-OVX+RollPCA(1)":
                f_path = os.path.join(RESULTS_DIR, f"{FILE_PREFIX}{model_name}_{target}.csv")
            else:
                f_path = os.path.join(BASELINE_DIR, f"{FILE_PREFIX}{model_name}_{target}.csv")
            
            if not os.path.exists(f_path):
                print(f"    [Warning] File missing for {model_name}, skipping.")
                continue
            
            try:
                df = pd.read_csv(f_path)
                df['date'] = pd.to_datetime(df['date'])
                df = df.drop_duplicates(subset=['date'], keep='first')
                df.set_index('date', inplace=True)
                df.sort_index(inplace=True)
                
        
                mask = (df.index >= EVAL_START_DATE) & (df.index <= effective_end_date)
                df_slice = df.loc[mask].copy()
                
            
                df_slice.dropna(subset=['Actual', 'Pred'], inplace=True)
                
                if df_slice.empty or 'Actual' not in df_slice.columns or 'Pred' not in df_slice.columns:
                    continue

  
                loss_series = qlike_loss_series(df_slice['Actual'], df_slice['Pred'])
                

                if np.isnan(loss_series).any() or np.isinf(loss_series).any():
                    print(f"    [Warning] {model_name}: Contains NaN/Inf, skipping.")
                    continue
                
                loss_dict[model_name] = loss_series
                
            except Exception as e:
                print(f"    [Error] Failed to process {model_name}: {e}")

        if len(loss_dict) < 2:
            print(f"    [Error] Insufficient models available for comparison.\n")
            continue

    
        loss_df = pd.DataFrame(loss_dict)
        loss_df.dropna(inplace=True)
        
        if loss_df.empty:
            print(f"    [Error] No overlapping dates available.\n")
            continue
        
        print(f"    Loaded {len(loss_df.columns)} models, {len(loss_df)} OOS samples")
        

        avg_loss = loss_df.mean().sort_values()
        print(f"    Avg QLIKE (Lower is Better):")
        for model in avg_loss.index:
            print(f"      {model}: {avg_loss[model]:.6f}")
        

        models = loss_df.columns.tolist()
        n_models = len(models)
        
        print(f"\n    Pairwise DM Tests:")
        print(f"    {'Model A':<22} vs {'Model B':<22} | Mean_Diff | DM_Stat | P_Value | Result")
        print(f"    {'-'*100}")
        
        for i in range(n_models):
            for j in range(i + 1, n_models):
                model_a = models[i]
                model_b = models[j]
                
                loss_a = loss_df[model_a]
                loss_b = loss_df[model_b]
                
                mean_diff = loss_a.mean() - loss_b.mean()
                dm_stat, p_val = dm_test(loss_a, loss_b)
                
                if p_val < 0.01:
                    result = "***"
                elif p_val < 0.05:
                    result = "**"
                elif p_val < 0.10:
                    result = "*"
                else:
                    result = "NS"
                
                print(f"    {model_a:<22} vs {model_b:<22} | {mean_diff:9.6f} | {dm_stat:7.3f} | {p_val:7.4f} | {result}")
                
                all_dm_results.append({
                    "Horizon": target,
                    "Model_A": model_a,
                    "Model_B": model_b,
                    "Avg_QLIKE_A": loss_a.mean(),
                    "Avg_QLIKE_B": loss_b.mean(),
                    "Mean_Diff": mean_diff,
                    "DM_Stat": dm_stat,
                    "P_Value": p_val,
                    "Significant": result
                })
        
        print()

    if all_dm_results:
        final_df = pd.DataFrame(all_dm_results)
        output_path = os.path.join(RESULTS_DIR, "dm_test_pairwise_results.csv")
        final_df.to_csv(output_path, index=False)
        
        print(f"========================================================")
        print(f"   DM Test Summary (Statistically Significant)")
        print(f"========================================================")
        pd.options.display.float_format = "{:,.6f}".format
        
        for target in TARGETS:
            subset = final_df[final_df["Horizon"] == target]
            if subset.empty: 
                continue
            
            print(f"\n--- {target} ---")
            sig_subset = subset[subset["P_Value"] < 0.10]
            if not sig_subset.empty:
                cols = ["Model_A", "Model_B", "Mean_Diff", "DM_Stat", "P_Value", "Significant"]
                print(sig_subset[cols].to_string(index=False))
            else:
                print("    (No significant differences found at the 10% level)")
            
        print(f"\n✓ Pairwise DM results successfully saved to: {output_path}")
    else:
        print("\n[Error] No evaluation results generated.")